# Train a conditional RSGM: poses given keypoints

RSGM conditioned on the 3D positions of the 22 SMPL joints (`docs/rsgm_conditioning.md`), for inverse kinematics with and without occlusions (§4.2, §4.3 of the HuProSO3 paper). The reference is the authors' conditional flow `id_0384` (`explore/evaluation/conditional_flow_occlusion.ipynb`), which reproduces Tab. 3 of the paper on our data.

- **Data:** all frames of our AMASS (`~/data/AMASS_rotations`), the official train / valid split. The keypoints are computed from the poses on the fly in the loss (no change to the dataset).
- **Condition:** `condition = "3D"` (66 numbers), and `cond_mask = "random"` hides joints at random as the paper's training for occlusions. So one model serves full IK and every occlusion, as `id_0384`.
- **Model and training:** the settings of our best unconditional RSGM (round 1, trial 23 with AdamW, `rsgm_tuned_r1t23_adamw`: 3 × 1024 silu, β_f ≈ 9, lr ≈ 1e-3, batch 512, EMA 0.999, 100k steps), plus the condition MLP of the paper (one hidden layer of 64, 64 features).
- **Device:** mps. One step costs ~8 ms (unconditional ~5.4 ms): ~15-20 min for 100k steps.

The validation loss is the denoising loss of the EMA weights on the valid split with fixed times, noise and masks, so validations are comparable. The best checkpoint by it is kept (`keep_best`).

**One setting to watch: `loss_t_min`.** Trial 23 trains the score only for t ≥ 3.1·10⁻³, but the sampler runs down to eps = 2·10⁻⁴. Without a condition this did not matter. Given the keypoints, the poses are pinned down very precisely in 33 of the 57 directions (`docs/rsgm_conditioning.md`, section 6), so the score at small t decides the position error of the samples. `LOSS_T_MIN` is therefore a parameter; the first run keeps trial 23's value, and the quick check at the end shows whether the visible keypoints are reproduced to a few mm.

A finished run (a `run_*` folder with `history.json`) is not trained again; `RETRAIN = True` trains a new one (runs are never overwritten).

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import roma
import torch
from omegaconf import OmegaConf

from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS, AmassJoints, AmassJoints4Learning
from hp.data.conditions import keypoints
from hp.evaluation.data import load_reference
from hp.models.io import load_model
from hp.training import best_checkpoint, cli
from hp.training.config import load_config

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations"        # all frames (needs "data.load_dtype=float32" in OVERRIDES)
SPLIT = "original"                          # the official AMASS split
RUN_ROOT = "~/data/experiments"             # runs go to RUN_ROOT/RUN_NAME/run_0001, run_0002, ...
RUN_NAME = "rsgm_cond3d_t23_adamw"
DEVICE = "mps"                              # "cpu", "mps" or "cuda:0"
SEED = 10
RETRAIN = False                             # True: train a new run even if a finished one exists
CONDITION = "3D"                            # "3D" (inverse kinematics) or "2D" (uplifting)
COND_MASK = "random"                        # "random": hidden joints as the paper's occlusion training; "none": all visible
LOSS_T_MIN = 0.0030728873202715665          # trial 23; try 1e-3 (the default) if the visible keypoints are not reproduced
MAX_ITERATIONS = 100_000
# round 1, trial 23, with AdamW (rsgm_tuned_r1t23_adamw: test log-likelihood 138.5, the best unconditional RSGM)
TUNED = ["model.args.hidden=[1024, 1024, 1024]", "model.args.activation=silu", "model.args.beta_f=8.989556539013165",
         "model.args.weighting=std", "train.lr=0.0010569766304589611", "train.batch_size=512", "train.ema_decay=0.999",
         "train.scheduler=step", "train.optimizer=adamw"]
CONDITIONING = [f"model.args.condition={CONDITION}", f"model.args.cond_mask={COND_MASK}", "model.args.cond_hidden=64",
                "model.args.cond_features=64", f"model.args.loss_t_min={LOSS_T_MIN}"]
OVERRIDES = ["data.load_dtype=float32", "train.valid_interval=5000", f"train.max_iterations={MAX_ITERATIONS}", *TUNED, *CONDITIONING]
# the quick check at the end
N_CHECK = 500                               # valid poses
K = 10                                      # samples per pose ("mean of K", "best of K")
OCCLUSIONS = {                              # the occlusions of the paper (Tab. 3), by the keypoints they hide
    "none": [],
    "L (left leg)": ["LeftUpLeg", "LeftLeg", "LeftFoot"],
    "A+H (left arm and hand)": ["LeftArm", "LeftElbow", "LeftHand"],
    "S+UA (right shoulder, upper arm)": ["RightShoulder", "RightArm"],
}
N_LIKELIHOOD = 200                          # valid poses for the conditional log-likelihood (probability flow ODE)
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
argv = ["model=rsgm", f"run_root={RUN_ROOT}", f"run_name={RUN_NAME}", f"seed={SEED}", f"data.rotations={ROTATIONS}",
        f"data.split={SPLIT}", f"train.device={DEVICE}", *OVERRIDES]
cfg = load_config(argv)
print(OmegaConf.to_yaml(cfg))

train_frames = sum(torch.load(expand(ROTATIONS) / f"{d}.pt", mmap=True).shape[0] for d in ORIGINAL_AMASS_SPLITS["train"])
iterations, batch = cfg.train.max_iterations, cfg.train.batch_size
print(f"{iterations:,} iterations x batch {batch} = {iterations * batch / train_frames:.1f} epochs over {train_frames:,} training frames")

## Train

With `scripts/train_model.py` (`hp.training.cli.main`). The wall time is stored in `train_time.json` of the run folder.

In [ ]:
def finished_run(root):
    # The newest run folder below root that finished (has history.json and weights.pt), or None.
    runs = sorted(p for p in root.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / "weights.pt").is_file())
    return runs[-1] if runs else None


run_dir = None if RETRAIN else finished_run(expand(RUN_ROOT) / RUN_NAME)
if run_dir is not None:
    print(f"already trained: {run_dir}")
else:
    start = time.perf_counter()
    run_dir = cli.main(argv)
    seconds = time.perf_counter() - start
    (run_dir / "train_time.json").write_text(json.dumps({"seconds": seconds, "device": DEVICE}))
    print(f"trained in {seconds / 60:.1f} min")

## Loss curves

Training loss of every step (and its moving average) and the validation loss. Epochs over the training frames on the top axis.

In [ ]:
SMOOTH = 200  # steps of the moving average

history = json.loads((run_dir / "history.json").read_text())
loss = np.asarray(history["loss"], dtype=float)
steps = np.arange(1, len(loss) + 1)
smooth = np.convolve(loss, np.ones(SMOOTH) / SMOOTH, mode="valid")
per_epoch = train_frames / batch  # steps per epoch

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, loss, color="C0", alpha=0.15, lw=0.5)
ax.plot(steps[SMOOTH - 1:], smooth, color="C0", label=f"training (moving average of {SMOOTH})")
ax.plot(history["valid_iteration"], history["valid_loss"], "o-", color="C1", label="validation (EMA weights, fixed noise and masks)")
ax.set_ylim(np.percentile(loss, 1) * 0.9, np.percentile(loss, 99.5) * 1.1)
ax.set_xlabel("step")
ax.set_ylabel("denoising score matching loss")
ax.secondary_xaxis("top", functions=(lambda s: s / per_epoch, lambda e: e * per_epoch)).set_xlabel("epoch")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.loglog()
plt.show()

if history["valid_loss"]:
    best = int(np.argmin(history["valid_loss"]))
    print(f"validation loss: last {history['valid_loss'][-1]:.3f}, best {history['valid_loss'][best]:.3f} at step {history['valid_iteration'][best]:,}")

## Quick check

The best checkpoint (EMA weights) on `N_CHECK` valid poses, for every occlusion of `OCCLUSIONS`: `K` samples per pose given the visible keypoints, then

- **MPJPE** [mm] over the 22 keypoints and **MGEO** [rad] over the 19 joints, of one sample, of the mean pose of the K samples, and the best of K (the conventions of `conditional_flow_occlusion.ipynb`);
- **MPJPE of the visible keypoints**: how well the condition is met. It should be a few mm; if it is much larger, the score at small t is not precise enough (see `LOSS_T_MIN`).

The paper's Tab. 3 ("Ours, N = 10", test split) and the authors' flow on our test split are the yardsticks: L 13.9 | 0.165, A+H 30.2 | 0.201, S+UA 12.7 | 0.162. These numbers are on the valid split, so they are only a first look; the comparison on the test split comes with the evaluation notebook.

In [ ]:
def mean_rotation(poses):
    # the chordal mean (as scipy Rotation.mean) over dim 1 of poses [B, k, J, 3, 3] -> [B, J, 3, 3]
    q = roma.rotmat_to_unitquat(poses.double())
    top = torch.linalg.eigh(torch.einsum("bkji,bkjl->bjil", q, q)).eigenvectors[..., -1]
    return roma.unitquat_to_rotmat(top).float()


def condition_of(kp, hidden):
    # flattened keypoints [B, 66] with the joints `hidden` (indices 0-21) as NaN: not observed
    kp = kp.clone()
    kp[:, list(hidden)] = float("nan")
    return kp.reshape(len(kp), -1)


checkpoint = best_checkpoint(run_dir) or run_dir
model = load_model(checkpoint, device=DEVICE, weights="weights_ema")
print(f"evaluated: {checkpoint} (EMA weights), {model.num_parameters():,} parameters, condition {model.condition}, mask {model.cond_mask}")

valid = load_reference("valid", ROTATIONS, SPLIT)
gt = valid[torch.randperm(len(valid), generator=torch.Generator().manual_seed(SEED))[:N_CHECK]]
kp = keypoints(gt)
J = len(AmassJoints4Learning)
rows = {}
for occlusion, names in OCCLUSIONS.items():
    hidden = [AmassJoints[n].value for n in names]
    visible = [j for j in range(22) if j not in hidden]
    torch.manual_seed(SEED)
    start = time.perf_counter()
    s = model.sample(N_CHECK * K, condition_of(kp, hidden).repeat_interleave(K, 0)).cpu().reshape(N_CHECK, K, J, 3, 3)
    seconds = time.perf_counter() - start
    err = (keypoints(s.reshape(-1, J, 3, 3)).reshape(N_CHECK, K, 22, 3) - kp[:, None]).norm(dim=-1) * 1000   # [N, K, 22]
    mgeo = roma.rotmat_geodesic_distance(s, gt[:, None]).mean(-1)                                           # [N, K]
    m = mean_rotation(s)
    rows[occlusion] = dict(
        mpjpe_1=err[:, 0].mean(), mgeo_1=mgeo[:, 0].mean(),
        mpjpe_mean=((keypoints(m) - kp).norm(dim=-1) * 1000).mean(), mgeo_mean=roma.rotmat_geodesic_distance(m, gt).mean(),
        mpjpe_best=err.mean(-1).min(1).values.mean(), mgeo_best=mgeo.min(1).values.mean(),
        visible=err[:, :, visible].mean(), seconds=seconds)

print(f"\n{N_CHECK} valid poses, {K} samples each. MPJPE [mm] | MGEO [rad]\n")
print(f"{'occlusion':34s} {'1 sample':>15s} {f'mean of {K}':>15s} {f'best of {K}':>15s} {'visible MPJPE':>14s} {'s / 1000':>9s}")
for occlusion, r in rows.items():
    print(f"{occlusion:34s} {r['mpjpe_1']:5.1f} | {r['mgeo_1']:.3f} {r['mpjpe_mean']:7.1f} | {r['mgeo_mean']:.3f} "
          f"{r['mpjpe_best']:7.1f} | {r['mgeo_best']:.3f} {r['visible']:11.1f} mm {r['seconds'] / (N_CHECK * K) * 1000:9.2f}")

The conditional log-likelihood log p(pose | keypoints) of `N_LIKELIHOOD` valid poses, all joints visible and with the A+H occlusion. It must be finite. Its size is set by the smoothing at eps, not by the data alone (`docs/rsgm_conditioning.md`, section 6), so it is not compared with the flow's.

In [ ]:
poses, kp_ll = gt[:N_LIKELIHOOD], kp[:N_LIKELIHOOD]
for occlusion in ("none", "A+H (left arm and hand)"):
    hidden = [AmassJoints[n].value for n in OCCLUSIONS[occlusion]]
    start = time.perf_counter()
    with torch.no_grad():
        ll = torch.cat([model.log_prob(poses[i:i + 100].to(model.device), condition_of(kp_ll[i:i + 100], hidden))
                        for i in range(0, len(poses), 100)]).cpu()
    print(f"{occlusion:28s} log p(pose | keypoints): mean {ll.mean():7.2f}, median {ll.median():7.2f}, "
          f"{(~torch.isfinite(ll)).sum()} non-finite, {(time.perf_counter() - start) / len(poses) * 1000:.0f} ms per pose")